# ICESat-2 Validation of Sentinel-2 SDB — Algarve reefs

Validates each Stumpf satellite-derived bathymetry (SDB) raster against ICESat-2
ATL08 photon elevations -> per-scene **RMSE / bias / Pearson-r** table
(`validation_summary.csv`), the ground-truth check missing from the orchestrator report.

**Pipeline:** discover `bathy_s2_stumpf_<YYYYMMDD>.tif` (depth maps only, not terrain
derivatives) -> normalize depth sign -> `run_icesat2_validation()` (auto bbox+date,
ATL08 search, SDB sampling, metrics) -> aggregate CSV + RMSE chart -> flag failures.

> **Sign-convention note.** These SDB rasters store depth as *negative* (e.g. -4.3 m),
> but `icesat2_validation` compares against *positive* photon depth and samples the
> raster raw. We therefore normalize each raster to positive depth before validating.
> The production orchestrator path needs the same fix in `_sample_sdb_at_points` /
> the depth-map writer, or real RMSE will be off by ~2x depth.

**Modes:** `DRY_RUN=True` -> synthetic photons, no network (laptop). `DRY_RUN=False`
-> real OpenAltimetry ATL08 (run on the **DGT VM**).

In [ ]:
# Config
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SDB_ROOT    = PROJECT_ROOT / "reef_Output_Master"
OUTPUT_DIR  = PROJECT_ROOT / "outputs" / "icesat2_validation"
NORM_DIR    = OUTPUT_DIR / "_normalized"
SUMMARY_CSV = OUTPUT_DIR / "validation_summary.csv"
SUMMARY_PNG = OUTPUT_DIR / "validation_rmse.png"

SEARCH_DAYS = 90
RMSE_PASS_M = 2.0
DRY_RUN     = True   # False on the DGT VM for real OpenAltimetry validation

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT, "| DRY_RUN:", DRY_RUN)

In [ ]:
# Discover depth rasters (exclude terrain derivatives: _slope_deg/_bpi/_tri/_curvature)
import sys, re, math, json
import numpy as np, pandas as pd, rasterio
from pyproj import Transformer

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import icesat2_validation as iv

_DEPTH_RE = re.compile(r"bathy_s2_stumpf_\d{8}\.tif$")
rasters = sorted(p for p in SDB_ROOT.glob("**/bathy_s2_stumpf_*.tif")
                 if _DEPTH_RE.search(p.name))
print(f"{len(rasters)} pure-depth rasters (derivatives excluded)")
for r in rasters[:12]:
    print("  ", r.relative_to(PROJECT_ROOT))

In [ ]:
# Normalize depth sign -> positive metres (writes a corrected copy under _normalized/)
def normalize_depth_raster(path):
    """Return a path whose values are POSITIVE depth (m). If the source median is
    negative, write a sign-flipped copy; else return the source unchanged."""
    with rasterio.open(str(path)) as src:
        arr = src.read(1).astype("float32")
        prof = src.profile
        nod = src.nodata
    fin = np.isfinite(arr)
    if nod is not None:
        fin &= (arr != nod)
    if not fin.any():
        return path, 0
    med = float(np.median(arr[fin]))
    if med >= 0:
        return path, int(fin.sum())                      # already positive
    out = arr.copy()
    out[fin] = -arr[fin]                                 # flip sign on valid pixels
    dst_path = NORM_DIR / (path.stem + "_pos.tif")
    prof.update(dtype="float32")
    with rasterio.open(str(dst_path), "w", **prof) as dst:
        dst.write(out, 1)
    return dst_path, int(fin.sum())

In [ ]:
# Synthetic-photon mock (DRY_RUN only) — positive-depth rasters
from contextlib import contextmanager
from unittest.mock import patch

def _synthetic_photons_for(depth_map_path, n=60, offset_m=0.4, seed=7):
    rng = np.random.default_rng(seed)
    with rasterio.open(str(depth_map_path)) as src:
        arr = src.read(1).astype("float32"); nod = src.nodata
        valid = np.isfinite(arr) & (arr > 0)
        if nod is not None:
            valid &= (arr != nod)
        if not valid.any():
            return []
        rows, cols = np.where(valid)
        tf = Transformer.from_crs(src.crs.to_epsg(), 4326, always_xy=True)
        pick = rng.choice(len(rows), size=min(n, len(rows)), replace=False)
        photons = []
        for i in pick:
            x, y = src.xy(int(rows[i]), int(cols[i]))
            lon, lat = tf.transform(x, y)
            depth = float(arr[rows[i], cols[i]]) + offset_m + rng.standard_normal() * 0.3
            photons.append({"lat": lat, "lon": lon, "h": -depth})  # ICESat h negative below surface
        return photons

@contextmanager
def maybe_mock(depth_map_path):
    if DRY_RUN:
        with patch.object(iv, "_search_atl08", return_value=_synthetic_photons_for(depth_map_path)):
            yield
    else:
        yield

In [ ]:
# Run validation per scene
rows = []
for r in rasters:
    rel = str(r.relative_to(PROJECT_ROOT))
    norm_path, n_valid_px = normalize_depth_raster(r)
    out = OUTPUT_DIR / r.parent.name / r.stem
    try:
        with maybe_mock(norm_path):
            rep = iv.run_icesat2_validation(depth_map_path=norm_path, output_dir=out,
                                            search_days=SEARCH_DAYS)
    except Exception as e:
        rep = {"status": "error", "reason": str(e)}
    rep = rep or {"status": "none"}
    rows.append({"raster": rel, "site": r.parent.name, "scene_date": rep.get("scene_date"),
                 "status": rep.get("status"), "rmse_m": rep.get("rmse_m"),
                 "bias_m": rep.get("bias_m"), "mae_m": rep.get("mae_m"),
                 "pearson_r": rep.get("pearson_r"), "n_colocated": rep.get("n_colocated"),
                 "reason": rep.get("reason")})
    print(f"{rep.get('status'):>8}  rmse={rep.get('rmse_m')}  n={rep.get('n_colocated')}  {rel}")

df = pd.DataFrame(rows)
print(f"\n{len(df)} scenes; {(df.status=='ok').sum()} returned metrics")

In [ ]:
# Summary table + pass/fail
ok = df[df.status == "ok"].copy()
if not ok.empty:
    ok["passes_rmse"] = ok["rmse_m"] < RMSE_PASS_M
    df = df.merge(ok[["raster", "passes_rmse"]], on="raster", how="left")
df.to_csv(SUMMARY_CSV, index=False)
print("Wrote", SUMMARY_CSV)
if not ok.empty:
    print(f"Median RMSE {ok.rmse_m.median():.2f} m | pass(<{RMSE_PASS_M}m): "
          f"{int(ok.rmse_m.lt(RMSE_PASS_M).sum())}/{len(ok)}")
    fail = ok[ok.rmse_m >= RMSE_PASS_M]
    if not fail.empty:
        print("\nFAILING RMSE bound:")
        print(fail[["raster","rmse_m","bias_m","n_colocated"]].to_string(index=False))
df

In [ ]:
# RMSE bar chart
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
if not ok.empty:
    o = ok.sort_values("rmse_m")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4*len(o))))
    cols = ["#2a9d8f" if v < RMSE_PASS_M else "#e76f51" for v in o.rmse_m]
    ax.barh(o["raster"].str.replace("reef_Output_Master/","",regex=False), o.rmse_m, color=cols)
    ax.axvline(RMSE_PASS_M, ls="--", c="k", lw=1, label=f"pass {RMSE_PASS_M} m")
    ax.set_xlabel("RMSE (m) — SDB vs ICESat-2 ATL08")
    ax.set_title("ICESat-2 validation RMSE per SDB scene" + ("  [DRY RUN]" if DRY_RUN else ""))
    ax.legend(); plt.tight_layout(); fig.savefig(SUMMARY_PNG, dpi=130)
    print("Wrote", SUMMARY_PNG); plt.show()
else:
    print("No 'ok' scenes — on the VM check OpenAltimetry connectivity / widen SEARCH_DAYS.")

## Running on the DGT VM
1. `git pull` + `pip install -r requirements.txt`.
2. Set `DRY_RUN = False`, run all cells (needs VM outbound network for OpenAltimetry).
3. Outputs in `outputs/icesat2_validation/`: `validation_summary.csv`, `validation_rmse.png`,
   per-scene `validation_report.json`.

**Thesis (Ch. 5):** report median RMSE and flag sites over `RMSE_PASS_M`. Low `n_colocated`
-> widen `SEARCH_DAYS` (coastal ATL08 is sparse).

**Follow-up fix:** reconcile the depth sign in `src/icesat2_validation._sample_sdb_at_points`
(or the depth-map writer) so the production orchestrator path doesn't need this notebook's
normalization workaround.